# Alpamayo loop worker -- bedient die CARLA-Zeitlupen-Schleife

Laedt Alpamayo-1.5 einmal (M0-v23-Konfiguration: NF4-Backbone, FP16-Expert, KV-lokaler Split auf 2x T4) und wartet dann auf Jobs im privaten HF-Dataset-Repo `<hf-nutzer>/acarla-loop` (geschrieben von `scripts/run_closed_loop_alpamayo.py`). Jeder Job = ein SensorPacket (4 Kameras x 4 Frames + 16 Ego-Posen), jede Antwort = ein PlanResult. Heartbeat unter `worker/heartbeat.json`. Beendet sich nach 25 min ohne Jobs (GPU-Quote) oder wenn `worker/STOP` existiert.

Start nur ueber **Save & Run All** im Browser (das Secret `huggingface` muss angehaengt sein; ein API-Push verliert die Secret-Bindung).

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys

MODEL_ID = 'nvidia/Alpamayo-1.5-10B'
UPSTREAM_URL = 'https://github.com/NVlabs/alpamayo1.5.git'
UPSTREAM_COMMIT = '24179cfa8b2eeaf775e9e21698b23af0f899522d'
WORK = Path('/kaggle/temp/alpamayo_m0')
CACHE = Path('/kaggle/temp/huggingface')
RESULT = Path('/kaggle/working/m0_results/m0_kaggle_attempt.json')
WORK.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
RESULT.parent.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
os.environ['UV_LINK_MODE'] = 'copy'

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
secret_source = 'environment' if hf_token else None
secret_errors = {}
if not hf_token:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    for secret_label in (
        'HF_TOKEN',
        'HUGGING_FACE_HUB_TOKEN',
        'HUGGINGFACE_TOKEN',
        'HF_ACCESS_TOKEN',
        'huggingface',
    ):
        try:
            candidate = secret_client.get_secret(secret_label)
        except Exception as exc:
            secret_errors[secret_label] = f'{type(exc).__name__}: {exc}'
            continue
        if candidate:
            hf_token = candidate
            secret_source = f'kaggle:{secret_label}'
            break
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
os.environ['M0_SECRET_SOURCE'] = secret_source or 'none'

print(json.dumps({
    'python_base': platform.python_version(),
    'hf_token_present': bool(hf_token),
    'hf_token_source': secret_source,
    'secret_lookup_errors': secret_errors,
    'working_free_gib': round(shutil.disk_usage('/kaggle/working').free / 1024**3, 1),
    'temp_free_gib': round(shutil.disk_usage('/kaggle/temp').free / 1024**3, 1),
}, indent=2))
if not hf_token:
    print('WARNING: HF_TOKEN is missing. Model files are public, but the PhysicalAI dataset may require accepted access and authentication.')

In [ ]:
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, check=True, env=os.environ.copy())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.7'], check=True)
UV = shutil.which('uv')
assert UV, 'uv executable was not installed'
REPO = WORK / 'alpamayo1.5'
if not (REPO / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', UPSTREAM_URL, str(REPO)])
run(['git', 'fetch', 'origin', UPSTREAM_COMMIT, '--depth', '1'], cwd=REPO)
run(['git', 'checkout', '--detach', UPSTREAM_COMMIT], cwd=REPO)
run([UV, 'sync', '--no-install-package', 'flash-attn'], cwd=REPO)
PYTHON312 = REPO / '.venv/bin/python'
assert PYTHON312.exists(), 'Python 3.12 environment was not created'
# v20b: explizit in das Runner-Env installieren. Ohne --python loeste uv ein anderes Env auf,
# und der Runner fiel still auf FP16 zurueck (bitsandbytes_import_error in Version 22).
run([UV, 'pip', 'install', '--python', str(PYTHON312), 'bitsandbytes>=0.45.0'], cwd=REPO)
run([str(PYTHON312), '-c', "import bitsandbytes, torch; print({'bitsandbytes': bitsandbytes.__version__, 'cuda': torch.version.cuda})"])
run([str(PYTHON312), '-c', "import platform, torch, transformers; print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__})"])

In [ ]:
import os
from pathlib import Path
PKG = Path('/kaggle/working/acarla_pkg')
files = {'acarla/__init__.py': '', 'acarla/types.py': '"""Shared data contracts between the CARLA adapter, the Alpamayo model wrapper,\nthe controller, and the trace recorder.\n\nThis module has no dependency outside numpy and the standard library. It does\nnot import `carla` or any Alpamayo package -- it only describes the shapes of\ndata that cross those boundaries.\n\nThree top-level structures:\n\n- `SensorPacket`: what the CARLA adapter hands to the model wrapper each tick.\n- `PlanResult`: what the model wrapper hands back.\n- `TraceFrame`: one JSONL record written by the recorder per simulation tick.\n  `RunHeader` is the first line of every trace file.\n\nAll array-shaped fields are validated in `__post_init__`. Violations raise\n`ValueError` with the expected and actual shape/dtype spelled out.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\nimport numpy as np\n\n# ---------------------------------------------------------------------------\n# Module constants -- the single source of truth for fixed history/plan sizes.\n# ---------------------------------------------------------------------------\n\nN_HISTORY_FRAMES = 4\n"""Number of past camera frames kept per camera in a `SensorPacket`."""\n\nN_EGO_WAYPOINTS = 16\n"""Number of past ego poses kept in a `SensorPacket`."""\n\nN_PLAN_WAYPOINTS = 64\n"""Number of future waypoints produced by the model in a `PlanResult`."""\n\n_ROTATION_TOL = 1e-4\n\nSAMPLE_DT = 0.1\n"""Seconds between consecutive samples that the model implicitly assumes.\n\nAudit finding (docs/alpamayo_api_findings.md, "Konsequenzen fuer den\nProjektplan" #3): the delta tokenizer discards timestamps entirely\n(third_party/alpamayo1.5/src/alpamayo1_5/tokenizers/delta_tokenizer.py:71)\nand the model implicitly treats every step as exactly 100 ms apart.\nTimestamps that do not sit on this 10 Hz grid are silently misinterpreted\nby the model rather than rejected by it -- so this contract must reject\nthem instead.\n"""\n\nSAMPLE_DT_TOLERANCE = 1e-3\n"""Allowed deviation (seconds) from `SAMPLE_DT` between consecutive samples.\n\nSee `SAMPLE_DT` docstring for why this is enforced here rather than left\nto the model.\n"""\n\nALPAMAYO_CAMERA_INDICES: dict[str, int] = {\n    "cross_left": 0,\n    "front_wide": 1,\n    "cross_right": 2,\n    "front_tele": 6,\n}\n"""Integer camera indices the upstream model identifies cameras by.\n\nAudit finding (docs/alpamayo_api_findings.md, "Konsequenzen fuer den\nProjektplan" #4): the model has no notion of camera *names* -- it indexes\ncameras by integer and sorts by ascending index for a consistent ordering\n(third_party/alpamayo1.5/src/alpamayo1_5/load_physical_aiavdataset.py:81-89,\n198-202). These are the four default camera indices.\n"""\n\nALPAMAYO_CAMERA_PROMPT_NAMES: dict[str, str] = {\n    "cross_left": "Front left camera",\n    "front_wide": "Front camera",\n    "cross_right": "Front right camera",\n    "front_tele": "Front telephoto camera",\n}\n"""Clear-text camera names used in the model prompt.\n\nMaps the same contract keys as `ALPAMAYO_CAMERA_INDICES` to the strings\nthe prompt template expects (third_party/alpamayo1.5/src/alpamayo1_5/helper.py:27-35).\n"""\n\n\n# ---------------------------------------------------------------------------\n# Validation helpers\n# ---------------------------------------------------------------------------\n\n\ndef _check_ndarray(\n    arr: Any,\n    name: str,\n    expected_shape: tuple[int | None, ...],\n    dtype: np.dtype | type,\n) -> None:\n    """Validate that `arr` is an ndarray with `expected_shape` and `dtype`.\n\n    A `None` entry in `expected_shape` matches any size in that dimension.\n    Raises `ValueError` naming the expected and actual shape/dtype on failure.\n    """\n    if not isinstance(arr, np.ndarray):\n        raise ValueError(f"{name}: expected numpy.ndarray, got {type(arr).__name__}")\n\n    actual_shape = arr.shape\n    if len(actual_shape) != len(expected_shape) or not all(\n        e is None or e == a for e, a in zip(expected_shape, actual_shape, strict=True)\n    ):\n        expected_str = tuple("*" if e is None else e for e in expected_shape)\n        raise ValueError(\n            f"{name}: expected shape {expected_str}, got {actual_shape}"\n        )\n\n    if arr.dtype != np.dtype(dtype):\n        raise ValueError(\n            f"{name}: expected dtype {np.dtype(dtype)}, got {arr.dtype}"\n        )\n\n\ndef _check_strictly_increasing(values: list[float], name: str) -> None:\n    """Validate that `values` is strictly monotonically increasing.\n\n    Enforces the "oldest frame first" convention used throughout this\n    project. A violation here is the most dangerous failure mode: it means\n    history and future would be silently swapped.\n    """\n    for i in range(1, len(values)):\n        if not values[i] > values[i - 1]:\n            raise ValueError(\n                f"{name}: timestamps must be strictly increasing "\n                f"(oldest first), got {values[i - 1]} followed by {values[i]} "\n                f"at index {i}"\n            )\n\n\ndef _check_uniform_dt(values: list[float], name: str) -> None:\n    """Validate that consecutive differences in `values` all equal `SAMPLE_DT`\n    within `SAMPLE_DT_TOLERANCE`.\n\n    The model never sees these timestamps -- it assumes a fixed 100 ms step\n    between samples (see `SAMPLE_DT`). A spacing that drifts from that grid\n    is not detected by the model; it is silently reinterpreted as exactly\n    100 ms, which corrupts the implied ego/camera motion. This check turns\n    that silent misinterpretation into an explicit rejection here instead.\n    """\n    for i in range(1, len(values)):\n        actual_dt = values[i] - values[i - 1]\n        if abs(actual_dt - SAMPLE_DT) > SAMPLE_DT_TOLERANCE:\n            raise ValueError(\n                f"{name}: non-uniform sampling at index {i}: actual delta "\n                f"{actual_dt} does not match expected delta {SAMPLE_DT} "\n                f"within tolerance {SAMPLE_DT_TOLERANCE} -- the model does "\n                "not see timestamps and would silently treat this step as "\n                "exactly 100 ms, corrupting the implied motion"\n            )\n\n\ndef alpamayo_camera_order(names: Iterable[str]) -> list[str]:\n    """Return `names` sorted by ascending Alpamayo camera index.\n\n    This ordering is part of the model contract, not a convenience: the\n    upstream loader sorts cameras by integer index before stacking them\n    (third_party/alpamayo1.5/src/alpamayo1_5/load_physical_aiavdataset.py:\n    81-89, 198-202), so callers must not rely on their own call order.\n\n    Raises `ValueError` naming the allowed camera names if any entry in\n    `names` is not a known Alpamayo camera name.\n    """\n    names = list(names)\n    unknown = [n for n in names if n not in ALPAMAYO_CAMERA_INDICES]\n    if unknown:\n        raise ValueError(\n            f"alpamayo_camera_order: unknown camera name(s) {unknown}, "\n            f"allowed names are {sorted(ALPAMAYO_CAMERA_INDICES)}"\n        )\n    return sorted(names, key=lambda n: ALPAMAYO_CAMERA_INDICES[n])\n\n\ndef assert_valid_rotation(m: np.ndarray, name: str) -> None:\n    """Validate that `m` holds one or more valid rotation matrices.\n\n    `m` must have shape (..., 3, 3). Each trailing 3x3 block must be\n    orthonormal (`R @ R.T ≈ I`) with determinant `≈ +1`, tolerance 1e-4.\n    Raises `ValueError` on failure.\n    """\n    if not isinstance(m, np.ndarray) or m.ndim < 2 or m.shape[-2:] != (3, 3):\n        actual = m.shape if isinstance(m, np.ndarray) else type(m).__name__\n        raise ValueError(f"{name}: expected shape (..., 3, 3), got {actual}")\n\n    mats = m.reshape(-1, 3, 3)\n    identity = np.eye(3, dtype=mats.dtype)\n    for idx, r in enumerate(mats):\n        if not np.allclose(r @ r.T, identity, atol=_ROTATION_TOL):\n            raise ValueError(\n                f"{name}: rotation matrix at index {idx} is not orthonormal "\n                f"(R @ R.T deviates from identity by more than {_ROTATION_TOL})"\n            )\n        det = np.linalg.det(r)\n        if abs(det - 1.0) > _ROTATION_TOL:\n            raise ValueError(\n                f"{name}: rotation matrix at index {idx} has determinant "\n                f"{det}, expected +1 (tolerance {_ROTATION_TOL})"\n            )\n\n\n# ---------------------------------------------------------------------------\n# JSON (de)serialization helpers -- numpy arrays <-> nested lists, dtype-preserving\n# ---------------------------------------------------------------------------\n\n\ndef _ndarray_to_json(arr: np.ndarray) -> dict[str, Any]:\n    return {"dtype": str(arr.dtype), "data": arr.tolist()}\n\n\ndef _ndarray_from_json(d: dict[str, Any]) -> np.ndarray:\n    return np.array(d["data"], dtype=np.dtype(d["dtype"]))\n\n\n# ---------------------------------------------------------------------------\n# SensorPacket -- adapter -> model input\n# ---------------------------------------------------------------------------\n\n\n@dataclass\nclass SensorPacket:\n    """One model-input packet assembled by the CARLA adapter.\n\n    `images` holds `N_HISTORY_FRAMES` frames per camera, oldest frame first.\n    `ego_translation`/`ego_rotation` hold `N_EGO_WAYPOINTS` past ego poses,\n    expressed in the ego frame of the most recent frame, oldest first.\n    """\n\n    frame_id: int\n    sim_time: float\n    images: dict[str, np.ndarray]\n    image_timestamps: dict[str, list[float]]\n    ego_translation: np.ndarray\n    ego_rotation: np.ndarray\n    ego_timestamps: list[float]\n    command: str\n    nav_guidance: str | None\n\n    def __post_init__(self) -> None:\n        if self.images.keys() != self.image_timestamps.keys():\n            raise ValueError(\n                "images and image_timestamps must have the same camera keys, "\n                f"got images={sorted(self.images.keys())} "\n                f"image_timestamps={sorted(self.image_timestamps.keys())}"\n            )\n\n        if not self.images:\n            raise ValueError(\n                "images: must be a non-empty subset of the known Alpamayo "\n                f"camera names, got no cameras (allowed names are "\n                f"{sorted(ALPAMAYO_CAMERA_INDICES)})"\n            )\n        unknown_cams = set(self.images.keys()) - set(ALPAMAYO_CAMERA_INDICES)\n        if unknown_cams:\n            raise ValueError(\n                "images: keys must be a subset of the known Alpamayo camera "\n                f"names, got unknown keys {sorted(unknown_cams)} "\n                f"(allowed names are {sorted(ALPAMAYO_CAMERA_INDICES)})"\n            )\n\n        for cam, img in self.images.items():\n            _check_ndarray(\n                img,\n                f"images[{cam!r}]",\n                (N_HISTORY_FRAMES, None, None, 3),\n                np.uint8,\n            )\n            ts = self.image_timestamps[cam]\n            if len(ts) != N_HISTORY_FRAMES:\n                raise ValueError(\n                    f"image_timestamps[{cam!r}]: expected length "\n                    f"{N_HISTORY_FRAMES}, got {len(ts)}"\n                )\n            _check_strictly_increasing(ts, f"image_timestamps[{cam!r}]")\n            _check_uniform_dt(ts, f"image_timestamps[{cam!r}]")\n\n        _check_ndarray(\n            self.ego_translation,\n            "ego_translation",\n            (N_EGO_WAYPOINTS, 3),\n            np.float32,\n        )\n        _check_ndarray(\n            self.ego_rotation,\n            "ego_rotation",\n            (N_EGO_WAYPOINTS, 3, 3),\n            np.float32,\n        )\n        assert_valid_rotation(self.ego_rotation, "ego_rotation")\n\n        if len(self.ego_timestamps) != N_EGO_WAYPOINTS:\n            raise ValueError(\n                f"ego_timestamps: expected length {N_EGO_WAYPOINTS}, "\n                f"got {len(self.ego_timestamps)}"\n            )\n        _check_strictly_increasing(self.ego_timestamps, "ego_timestamps")\n        _check_uniform_dt(self.ego_timestamps, "ego_timestamps")\n\n\n# ---------------------------------------------------------------------------\n# PlanResult -- model output\n# ---------------------------------------------------------------------------\n\n\n@dataclass\nclass PlanResult:\n    """The model\'s planned future trajectory for one frame."""\n\n    frame_id: int\n    waypoints_xyz: np.ndarray\n    waypoints_rot: np.ndarray\n    reasoning: str | None\n    inference_ms: float\n    model_config_hash: str\n\n    def __post_init__(self) -> None:\n        _check_ndarray(\n            self.waypoints_xyz,\n            "waypoints_xyz",\n            (N_PLAN_WAYPOINTS, 3),\n            np.float32,\n        )\n        _check_ndarray(\n            self.waypoints_rot,\n            "waypoints_rot",\n            (N_PLAN_WAYPOINTS, 3, 3),\n            np.float32,\n        )\n        assert_valid_rotation(self.waypoints_rot, "waypoints_rot")\n\n    def to_json_dict(self) -> dict[str, Any]:\n        """Serialize to a JSON-compatible dict (numpy arrays -> nested lists)."""\n        return {\n            "frame_id": self.frame_id,\n            "waypoints_xyz": _ndarray_to_json(self.waypoints_xyz),\n            "waypoints_rot": _ndarray_to_json(self.waypoints_rot),\n            "reasoning": self.reasoning,\n            "inference_ms": self.inference_ms,\n            "model_config_hash": self.model_config_hash,\n        }\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> PlanResult:\n        """Deserialize from a dict produced by `to_json_dict`."""\n        return PlanResult(\n            frame_id=d["frame_id"],\n            waypoints_xyz=_ndarray_from_json(d["waypoints_xyz"]),\n            waypoints_rot=_ndarray_from_json(d["waypoints_rot"]),\n            reasoning=d["reasoning"],\n            inference_ms=d["inference_ms"],\n            model_config_hash=d["model_config_hash"],\n        )\n\n\n# ---------------------------------------------------------------------------\n# Small named structures used inside TraceFrame -- no loose dicts.\n# ---------------------------------------------------------------------------\n\n\n@dataclass\nclass Pose:\n    """A rigid transform: translation + rotation matrix."""\n\n    translation: np.ndarray  # (3,) float32\n    rotation: np.ndarray  # (3, 3) float32\n\n    def __post_init__(self) -> None:\n        _check_ndarray(self.translation, "translation", (3,), np.float32)\n        _check_ndarray(self.rotation, "rotation", (3, 3), np.float32)\n        assert_valid_rotation(self.rotation, "rotation")\n\n    def to_json_dict(self) -> dict[str, Any]:\n        return {\n            "translation": _ndarray_to_json(self.translation),\n            "rotation": _ndarray_to_json(self.rotation),\n        }\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> Pose:\n        return Pose(\n            translation=_ndarray_from_json(d["translation"]),\n            rotation=_ndarray_from_json(d["rotation"]),\n        )\n\n\n@dataclass\nclass BoundingBox:\n    """An actor\'s 3D bounding box, in the actor\'s local frame."""\n\n    extent: np.ndarray  # (3,) float32, half-sizes\n    location: np.ndarray  # (3,) float32, offset from actor transform origin\n\n    def __post_init__(self) -> None:\n        _check_ndarray(self.extent, "extent", (3,), np.float32)\n        _check_ndarray(self.location, "location", (3,), np.float32)\n\n    def to_json_dict(self) -> dict[str, Any]:\n        return {\n            "extent": _ndarray_to_json(self.extent),\n            "location": _ndarray_to_json(self.location),\n        }\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> BoundingBox:\n        return BoundingBox(\n            extent=_ndarray_from_json(d["extent"]),\n            location=_ndarray_from_json(d["location"]),\n        )\n\n\n@dataclass\nclass ActorState:\n    """One non-ego actor\'s state at a given tick, in CARLA world coordinates."""\n\n    id: int\n    type_id: str\n    bounding_box: BoundingBox\n    transform: Pose\n    velocity: np.ndarray  # (3,) float32\n\n    def __post_init__(self) -> None:\n        _check_ndarray(self.velocity, "velocity", (3,), np.float32)\n\n    def to_json_dict(self) -> dict[str, Any]:\n        return {\n            "id": self.id,\n            "type_id": self.type_id,\n            "bounding_box": self.bounding_box.to_json_dict(),\n            "transform": self.transform.to_json_dict(),\n            "velocity": _ndarray_to_json(self.velocity),\n        }\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> ActorState:\n        return ActorState(\n            id=d["id"],\n            type_id=d["type_id"],\n            bounding_box=BoundingBox.from_json_dict(d["bounding_box"]),\n            transform=Pose.from_json_dict(d["transform"]),\n            velocity=_ndarray_from_json(d["velocity"]),\n        )\n\n\n@dataclass\nclass LaneSegment:\n    """One lane-geometry polyline near the ego vehicle, in world coordinates."""\n\n    lane_id: int\n    polyline: np.ndarray  # (N, 3) float32, N >= 2\n\n    def __post_init__(self) -> None:\n        _check_ndarray(self.polyline, "polyline", (None, 3), np.float32)\n        if self.polyline.shape[0] < 2:\n            raise ValueError(\n                f"polyline: expected at least 2 points, got {self.polyline.shape[0]}"\n            )\n\n    def to_json_dict(self) -> dict[str, Any]:\n        return {"lane_id": self.lane_id, "polyline": _ndarray_to_json(self.polyline)}\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> LaneSegment:\n        return LaneSegment(\n            lane_id=d["lane_id"], polyline=_ndarray_from_json(d["polyline"])\n        )\n\n\n@dataclass\nclass TrafficLightState:\n    """One traffic light\'s state at a given tick."""\n\n    id: int\n    state: str\n    position: np.ndarray  # (3,) float32\n\n    def __post_init__(self) -> None:\n        _check_ndarray(self.position, "position", (3,), np.float32)\n\n    def to_json_dict(self) -> dict[str, Any]:\n        return {\n            "id": self.id,\n            "state": self.state,\n            "position": _ndarray_to_json(self.position),\n        }\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> TrafficLightState:\n        return TrafficLightState(\n            id=d["id"], state=d["state"], position=_ndarray_from_json(d["position"])\n        )\n\n\n@dataclass\nclass ControlCommand:\n    """The control command applied at a given tick."""\n\n    steer: float\n    throttle: float\n    brake: float\n\n    def to_json_dict(self) -> dict[str, Any]:\n        return {"steer": self.steer, "throttle": self.throttle, "brake": self.brake}\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> ControlCommand:\n        return ControlCommand(\n            steer=d["steer"], throttle=d["throttle"], brake=d["brake"]\n        )\n\n\n# ---------------------------------------------------------------------------\n# RunHeader -- first line of every JSONL trace file\n# ---------------------------------------------------------------------------\n\n\n@dataclass\nclass RunHeader:\n    """Metadata written once, as the first line of a run\'s JSONL trace file."""\n\n    carla_version: str\n    map_name: str\n    seed_world: int\n    seed_traffic_manager: int\n    model_config_name: str\n    model_config_hash: str\n    git_sha: str\n    run_timestamp: str\n    cameras: list[str]\n    fixed_delta_seconds: float\n\n    def to_json_dict(self) -> dict[str, Any]:\n        return {\n            "carla_version": self.carla_version,\n            "map_name": self.map_name,\n            "seed_world": self.seed_world,\n            "seed_traffic_manager": self.seed_traffic_manager,\n            "model_config_name": self.model_config_name,\n            "model_config_hash": self.model_config_hash,\n            "git_sha": self.git_sha,\n            "run_timestamp": self.run_timestamp,\n            "cameras": list(self.cameras),\n            "fixed_delta_seconds": self.fixed_delta_seconds,\n        }\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> RunHeader:\n        return RunHeader(\n            carla_version=d["carla_version"],\n            map_name=d["map_name"],\n            seed_world=d["seed_world"],\n            seed_traffic_manager=d["seed_traffic_manager"],\n            model_config_name=d["model_config_name"],\n            model_config_hash=d["model_config_hash"],\n            git_sha=d["git_sha"],\n            run_timestamp=d["run_timestamp"],\n            cameras=list(d["cameras"]),\n            fixed_delta_seconds=d["fixed_delta_seconds"],\n        )\n\n\n# ---------------------------------------------------------------------------\n# TraceFrame -- one JSONL record per simulation tick\n# ---------------------------------------------------------------------------\n\n\n@dataclass\nclass TraceFrame:\n    """One recorded simulation tick. Images are never embedded here -- only\n    paths, relative to the run directory, into files written alongside the\n    trace."""\n\n    frame_id: int\n    sim_time: float\n    ego_pose_world: Pose\n    actors: list[ActorState]\n    lanes: list[LaneSegment]\n    traffic_lights: list[TrafficLightState]\n    plan: PlanResult | None\n    control: ControlCommand\n    image_paths: dict[str, list[str]] = field(default_factory=dict)\n\n    def to_json_dict(self) -> dict[str, Any]:\n        """Serialize to a JSON-compatible dict (numpy arrays -> nested lists)."""\n        return {\n            "frame_id": self.frame_id,\n            "sim_time": self.sim_time,\n            "ego_pose_world": self.ego_pose_world.to_json_dict(),\n            "actors": [a.to_json_dict() for a in self.actors],\n            "lanes": [lane.to_json_dict() for lane in self.lanes],\n            "traffic_lights": [t.to_json_dict() for t in self.traffic_lights],\n            "plan": self.plan.to_json_dict() if self.plan is not None else None,\n            "control": self.control.to_json_dict(),\n            "image_paths": {k: list(v) for k, v in self.image_paths.items()},\n        }\n\n    @staticmethod\n    def from_json_dict(d: dict[str, Any]) -> TraceFrame:\n        """Deserialize from a dict produced by `to_json_dict`."""\n        return TraceFrame(\n            frame_id=d["frame_id"],\n            sim_time=d["sim_time"],\n            ego_pose_world=Pose.from_json_dict(d["ego_pose_world"]),\n            actors=[ActorState.from_json_dict(a) for a in d["actors"]],\n            lanes=[LaneSegment.from_json_dict(lane) for lane in d["lanes"]],\n            traffic_lights=[\n                TrafficLightState.from_json_dict(t) for t in d["traffic_lights"]\n            ],\n            plan=PlanResult.from_json_dict(d["plan"]) if d["plan"] is not None else None,\n            control=ControlCommand.from_json_dict(d["control"]),\n            image_paths={k: list(v) for k, v in d["image_paths"].items()},\n        )\n', 'acarla/loop/__init__.py': '"""Slow-motion closed loop: CARLA (local) <-> Alpamayo worker (Kaggle).\n\nThe simulation is synchronous and simply does not tick while a plan is in\nflight, so wall-clock latency of the remote worker never becomes plan age.\n`codec` defines the wire format, `transport` the queue on a private Hugging\nFace dataset repository; both sides import only these two modules.\n"""\n', 'acarla/loop/codec.py': '"""Wire format between the local CARLA loop and the remote Alpamayo worker.\n\nA *job* carries exactly the raw numpy inputs `sensor_packet_to_model_inputs`\nproduces for one `SensorPacket`, with the 16 camera frames JPEG- or\nPNG-encoded so a job stays at a few megabytes instead of ~300 MB. A *plan*\nis the worker\'s answer: one `PlanResult` plus provenance, as JSON.\n\nFidelity note: the verified batch chain fed the F-Theta-remapped frames to the\nmodel uncompressed (`model_inputs.npz`). With `codec="jpg"` the remapped\nframes are compressed a second time (the CARLA render was already stored as\nJPEG q92); `codec="png"` reproduces the batch tensors bit-exactly at ~8x the\njob size. The processor downsamples 1080x1920 -> 320x576 anyway.\n"""\n\nfrom __future__ import annotations\n\nimport io\nimport json\nfrom typing import Any\n\nimport numpy as np\nfrom PIL import Image\n\nfrom acarla.types import (\n    N_HISTORY_FRAMES,\n    N_PLAN_WAYPOINTS,\n    PlanResult,\n)\n\nJOB_FORMAT = "acarla-loop-job-v1"\nPLAN_FORMAT = "acarla-loop-plan-v1"\nJPEG_QUALITY = 92  # same as the recorder writes to disk, so the second pass costs little\n\n\ndef encode_job(\n    model_inputs: dict[str, np.ndarray],\n    *,\n    run_id: str,\n    seq: int,\n    frame_id: int,\n    sim_time: float,\n    codec: str = "jpg",\n    quality: int = JPEG_QUALITY,\n) -> bytes:\n    """Serialize one packet\'s model inputs into an .npz byte string."""\n    if codec not in ("jpg", "png"):\n        raise ValueError(f"codec must be \'jpg\' or \'png\', got {codec!r}")\n    frames = np.asarray(model_inputs["image_frames"])\n    if frames.ndim != 5 or frames.shape[1] != N_HISTORY_FRAMES or frames.shape[2] != 3:\n        raise ValueError(f"image_frames: expected (N_cam, 4, 3, H, W), got {frames.shape}")\n    if frames.dtype != np.uint8:\n        raise ValueError(f"image_frames: expected uint8, got {frames.dtype}")\n    camera_indices = np.asarray(model_inputs["camera_indices"], dtype=np.int64)\n    if camera_indices.shape != (frames.shape[0],):\n        raise ValueError("camera_indices must have one entry per camera")\n\n    arrays: dict[str, np.ndarray] = {\n        "camera_indices": camera_indices,\n        "ego_history_xyz": np.asarray(model_inputs["ego_history_xyz"], dtype=np.float32),\n        "ego_history_rot": np.asarray(model_inputs["ego_history_rot"], dtype=np.float32),\n    }\n    for n in range(frames.shape[0]):\n        for k in range(N_HISTORY_FRAMES):\n            # CHW -> HWC only for the image encoder; `decode_job` inverts it.\n            hwc = np.ascontiguousarray(np.transpose(frames[n, k], (1, 2, 0)))\n            buf = io.BytesIO()\n            if codec == "jpg":\n                Image.fromarray(hwc).save(buf, "JPEG", quality=quality)\n            else:\n                Image.fromarray(hwc).save(buf, "PNG")\n            arrays[f"img_{n}_{k}"] = np.frombuffer(buf.getvalue(), dtype=np.uint8)\n    meta = {\n        "format": JOB_FORMAT,\n        "run_id": run_id,\n        "seq": int(seq),\n        "frame_id": int(frame_id),\n        "sim_time": float(sim_time),\n        "codec": codec,\n        "quality": int(quality) if codec == "jpg" else None,\n        "n_cameras": int(frames.shape[0]),\n        "height": int(frames.shape[3]),\n        "width": int(frames.shape[4]),\n        "camera_prompt_names": list(model_inputs.get("camera_prompt_names", [])),\n    }\n    arrays["meta_json"] = np.frombuffer(json.dumps(meta).encode("utf-8"), dtype=np.uint8)\n    out = io.BytesIO()\n    np.savez(out, **arrays)\n    return out.getvalue()\n\n\ndef decode_job(payload: bytes) -> dict[str, Any]:\n    """Inverse of `encode_job`; returns model inputs plus the `meta` dict."""\n    with np.load(io.BytesIO(payload)) as pk:\n        meta = json.loads(bytes(pk["meta_json"]).decode("utf-8"))\n        if meta.get("format") != JOB_FORMAT:\n            raise ValueError(f"unexpected job format {meta.get(\'format\')!r}")\n        n_cam, h, w = meta["n_cameras"], meta["height"], meta["width"]\n        frames = np.empty((n_cam, N_HISTORY_FRAMES, 3, h, w), dtype=np.uint8)\n        for n in range(n_cam):\n            for k in range(N_HISTORY_FRAMES):\n                img = Image.open(io.BytesIO(bytes(pk[f"img_{n}_{k}"]))).convert("RGB")\n                hwc = np.asarray(img, dtype=np.uint8)\n                if hwc.shape != (h, w, 3):\n                    raise ValueError(f"img_{n}_{k}: expected {(h, w, 3)}, got {hwc.shape}")\n                frames[n, k] = np.transpose(hwc, (2, 0, 1))  # back to CHW\n        return {\n            "image_frames": frames,\n            "camera_indices": pk["camera_indices"].astype(np.int64),\n            "ego_history_xyz": pk["ego_history_xyz"].astype(np.float32),\n            "ego_history_rot": pk["ego_history_rot"].astype(np.float32),\n            "meta": meta,\n        }\n\n\ndef encode_plan(\n    *,\n    run_id: str,\n    seq: int,\n    frame_id: int,\n    sim_time: float,\n    waypoints_xyz: np.ndarray,\n    waypoints_rot: np.ndarray,\n    reasoning: str | None,\n    inference_ms: float,\n    model_config_hash: str,\n    worker_session: str,\n    extra: dict[str, Any] | None = None,\n) -> str:\n    """Serialize a worker answer as JSON text (plain lists, no numpy types)."""\n    xyz = np.asarray(waypoints_xyz, dtype=np.float64)\n    rot = np.asarray(waypoints_rot, dtype=np.float64)\n    if xyz.shape != (N_PLAN_WAYPOINTS, 3) or rot.shape != (N_PLAN_WAYPOINTS, 3, 3):\n        raise ValueError(f"waypoints: got {xyz.shape} / {rot.shape}")\n    doc = {\n        "format": PLAN_FORMAT,\n        "run_id": run_id,\n        "seq": int(seq),\n        "frame_id": int(frame_id),\n        "sim_time": float(sim_time),\n        "waypoints_xyz": xyz.tolist(),\n        "waypoints_rot": rot.tolist(),\n        "reasoning": reasoning,\n        "inference_ms": float(inference_ms),\n        "model_config_hash": str(model_config_hash),\n        "finite": bool(np.isfinite(xyz).all() and np.isfinite(rot).all()),\n        "worker_session": worker_session,\n        "extra": extra or {},\n    }\n    return json.dumps(doc)\n\n\ndef decode_plan(text: str, *, expect_run_id: str, expect_seq: int) -> tuple[PlanResult, dict[str, Any]]:\n    """Parse a worker answer into a `PlanResult` (contract-validated) + raw doc."""\n    doc = json.loads(text)\n    if doc.get("format") != PLAN_FORMAT:\n        raise ValueError(f"unexpected plan format {doc.get(\'format\')!r}")\n    if doc.get("run_id") != expect_run_id or int(doc.get("seq", -1)) != expect_seq:\n        raise ValueError(\n            f"plan addressed to run={doc.get(\'run_id\')} seq={doc.get(\'seq\')}, "\n            f"expected run={expect_run_id} seq={expect_seq}"\n        )\n    if not doc.get("finite", False):\n        raise ValueError("worker marked the plan non-finite")\n    plan = PlanResult(\n        frame_id=int(doc["frame_id"]),\n        waypoints_xyz=np.asarray(doc["waypoints_xyz"], dtype=np.float32),\n        waypoints_rot=np.asarray(doc["waypoints_rot"], dtype=np.float32),\n        reasoning=doc.get("reasoning"),\n        inference_ms=float(doc["inference_ms"]),\n        model_config_hash=str(doc["model_config_hash"]),\n    )\n    return plan, doc\n', 'acarla/loop/transport.py': '"""Job/plan queue on a private Hugging Face *dataset* repository.\n\nWhy HF and not a socket: the local machine has no public address, the Kaggle\nkernel has outbound internet plus the Hugging Face secret it already needs\nfor the gated backbone weights. Every transfer is a small commit; both sides\npoll. Layout inside the repo::\n\n    runs/<run_id>/run.json            written by the local loop at start\n    runs/<run_id>/jobs/<seq:05d>.npz   one job per replan (see codec.encode_job)\n    runs/<run_id>/plans/<seq:05d>.json one answer per job (codec.encode_plan)\n    runs/<run_id>/DONE                local loop finished (worker skips the run)\n    worker/heartbeat.json             worker state, rewritten every minute\n    worker/STOP                       ask the worker to exit after the current job\n\nRate limit: the Hub allows 128 commits per hour *per repository*. One replan\ncosts one job commit and one plan commit, so a 1 s replan loop (~130 steps/h)\nwould exhaust a single repo twice over. Jobs and plans are therefore sharded\nover separate repos (`<base>-j0`, `<base>-j1`, `<base>-p0`, `<base>-p1`, seq\nmod 2), while `run.json`, `DONE`, `heartbeat.json`, `STOP` stay in the base\nrepo. A 429 is waited out (up to ~70 min) rather than treated as fatal.\n\nThe token is never handled here: `huggingface_hub` reads it from the\nenvironment (`HF_TOKEN`) or the local login (`hf auth login`).\n"""\n\nfrom __future__ import annotations\n\nimport io\nimport json\nimport random\nimport re\nimport tempfile\nimport time\nfrom pathlib import Path\nfrom typing import Any, Callable\n\ntry:  # optional: the directory backend and the unit tests need no hub client\n    from huggingface_hub import HfApi, hf_hub_download\n    from huggingface_hub.utils import HfHubHTTPError\nexcept ImportError:  # pragma: no cover\n    HfApi = hf_hub_download = None\n\n    class HfHubHTTPError(Exception):\n        pass\n\nDEFAULT_REPO = "acarla-loop"  # no namespace: resolved to <hf username>/acarla-loop at runtime\nN_SHARDS = 2\nRATE_LIMIT_WAIT_S = 70 * 60\n_SEQ_RE = re.compile(r"(\\d{5})\\.(npz|json)$")\n\n\ndef _status(exc: Exception) -> int | None:\n    return getattr(getattr(exc, "response", None), "status_code", None)\n\n\ndef _is_not_found(exc: Exception) -> bool:\n    if _status(exc) == 404:\n        return True\n    text = str(exc).lower()\n    return "404" in text or "not found" in text\n\n\ndef _retry(fn: Callable[[], Any], *, attempts: int = 6, what: str = "hf call") -> Any:\n    """Retry transient HTTP failures (5xx/network) with jittered backoff; wait\n    out a 429 commit-rate limit (128/h/repo) for up to RATE_LIMIT_WAIT_S."""\n    delay = 2.0\n    last: Exception | None = None\n    attempt = 0\n    rate_limited_since: float | None = None\n    while True:\n        attempt += 1\n        try:\n            return fn()\n        except HfHubHTTPError as exc:\n            status = _status(exc)\n            if status == 429:\n                rate_limited_since = rate_limited_since or time.time()\n                if time.time() - rate_limited_since > RATE_LIMIT_WAIT_S:\n                    raise RuntimeError(\n                        f"{what}: still rate limited after {RATE_LIMIT_WAIT_S // 60} min"\n                    ) from exc\n                print(f"  [{what}] HF rate limit (429) -- waiting 60 s "\n                      f"({(time.time() - rate_limited_since) / 60:.0f} min so far)", flush=True)\n                time.sleep(60.0)\n                attempt -= 1  # rate limiting does not count against the attempt budget\n                continue\n            if status is not None and status < 500 and status != 408:\n                raise\n            last = exc\n        except (ConnectionError, TimeoutError, OSError) as exc:\n            last = exc\n        if attempt >= attempts:\n            raise RuntimeError(f"{what} failed after {attempts} attempts: {last}") from last\n        time.sleep(delay + random.uniform(0.0, 1.0))\n        delay = min(delay * 2.0, 60.0)\n\n\nclass LoopQueue:\n    def __init__(self, repo_id: str = DEFAULT_REPO, cache_dir: str | Path | None = None) -> None:\n        if HfApi is None:\n            raise ImportError("huggingface_hub is required for the Hugging Face queue backend")\n        self.api = HfApi()\n        if "/" not in repo_id:  # HF user (say43) != Kaggle user (says43): never hard-code the namespace\n            repo_id = f"{self.api.whoami()[\'name\']}/{repo_id}"\n        self.repo_id = repo_id\n        self.job_repos = [f"{repo_id}-j{i}" for i in range(N_SHARDS)]\n        self.plan_repos = [f"{repo_id}-p{i}" for i in range(N_SHARDS)]\n        self.cache_dir = (\n            Path(cache_dir) if cache_dir else Path(tempfile.mkdtemp(prefix="acarla_loop_"))\n        )\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n\n    # -- setup -----------------------------------------------------------\n    def ensure_repo(self) -> None:\n        for repo in [self.repo_id, *self.job_repos, *self.plan_repos]:\n            _retry(\n                lambda repo=repo: self.api.create_repo(\n                    repo, repo_type="dataset", private=True, exist_ok=True\n                ),\n                what=f"create_repo {repo}",\n            )\n\n    def whoami(self) -> str:\n        return str(_retry(lambda: self.api.whoami(), what="whoami").get("name"))\n\n    # -- primitives ------------------------------------------------------\n    def _upload(self, path_in_repo: str, data: bytes, message: str, repo: str | None = None) -> None:\n        repo = repo or self.repo_id\n        _retry(\n            lambda: self.api.upload_file(\n                path_or_fileobj=io.BytesIO(data),\n                path_in_repo=path_in_repo,\n                repo_id=repo,\n                repo_type="dataset",\n                commit_message=message,\n            ),\n            what=f"upload {repo}:{path_in_repo}",\n        )\n\n    def _download(self, path_in_repo: str, *, mutable: bool, repo: str | None = None) -> bytes | None:\n        """Fetch a file\'s bytes; None if it does not exist. Mutable files\n        (heartbeat, run.json) bypass the content-addressed cache."""\n        repo = repo or self.repo_id\n\n        def go() -> bytes | None:\n            try:\n                local = hf_hub_download(\n                    repo,\n                    path_in_repo,\n                    repo_type="dataset",\n                    cache_dir=str(self.cache_dir),\n                    force_download=mutable,\n                )\n            except Exception as exc:\n                if _is_not_found(exc):\n                    return None\n                raise\n            return Path(local).read_bytes()\n\n        return _retry(go, what=f"download {path_in_repo}")\n\n    def _list(self, path_in_repo: str, repo: str | None = None) -> list[str]:\n        repo = repo or self.repo_id\n\n        def go() -> list[str]:\n            try:\n                entries = self.api.list_repo_tree(\n                    repo, path_in_repo=path_in_repo, repo_type="dataset", recursive=False\n                )\n                return [e.path for e in entries]\n            except Exception as exc:\n                if _is_not_found(exc):\n                    return []\n                raise\n\n        return _retry(go, what=f"list {path_in_repo}")\n\n    @staticmethod\n    def _seqs(paths: list[str]) -> list[int]:\n        out = []\n        for p in paths:\n            m = _SEQ_RE.search(p)\n            if m:\n                out.append(int(m.group(1)))\n        return sorted(out)\n\n    # -- local (producer) side -------------------------------------------\n    def start_run(self, run_id: str, info: dict[str, Any]) -> None:\n        doc = dict(info, run_id=run_id, status="active", started_at=time.time())\n        self._upload(\n            f"runs/{run_id}/run.json",\n            json.dumps(doc, indent=1).encode("utf-8"),\n            f"start run {run_id}",\n        )\n\n    def finish_run(self, run_id: str, summary: dict[str, Any] | None = None) -> None:\n        self._upload(\n            f"runs/{run_id}/DONE", json.dumps(summary or {}).encode("utf-8"), f"finish run {run_id}"\n        )\n\n    def put_job(self, run_id: str, seq: int, payload: bytes) -> None:\n        self._upload(f"runs/{run_id}/jobs/{seq:05d}.npz", payload, f"{run_id} job {seq}",\n                     repo=self.job_repos[seq % N_SHARDS])\n\n    def get_plan(self, run_id: str, seq: int) -> str | None:\n        data = self._download(f"runs/{run_id}/plans/{seq:05d}.json", mutable=False,\n                              repo=self.plan_repos[seq % N_SHARDS])\n        return None if data is None else data.decode("utf-8")\n\n    def heartbeat(self) -> dict[str, Any] | None:\n        data = self._download("worker/heartbeat.json", mutable=True)\n        return None if data is None else json.loads(data.decode("utf-8"))\n\n    def request_stop(self) -> None:\n        self._upload("worker/STOP", b"stop", "request worker stop")\n\n    def publish_code(self, acarla_src: str | Path) -> bool:\n        """Push `types.py` + `loop/*` as one JSON document so a freshly started\n        worker runs the same codec/transport as this side. Skipped (no commit)\n        when the published copy is already identical."""\n        root = Path(acarla_src)\n        files = {"acarla/__init__.py": ""}\n        for rel in ("types.py", "loop/__init__.py", "loop/codec.py", "loop/transport.py"):\n            files[f"acarla/{rel}"] = (root / rel).read_text(encoding="utf-8")\n        payload = json.dumps(files, sort_keys=True).encode("utf-8")\n        current = self._download("code/acarla_pkg.json", mutable=True)\n        if current == payload:\n            return False\n        self._upload("code/acarla_pkg.json", payload, "publish acarla loop package")\n        return True\n\n    # -- worker (consumer) side ------------------------------------------\n    def list_runs(self) -> list[str]:\n        return sorted(p.rsplit("/", 1)[-1] for p in self._list("runs"))\n\n    def run_is_done(self, run_id: str) -> bool:\n        return any(p.endswith("/DONE") for p in self._list(f"runs/{run_id}"))\n\n    def pending_jobs(self, run_id: str) -> list[int]:\n        jobs: set[int] = set()\n        for repo in self.job_repos:\n            jobs |= set(self._seqs(self._list(f"runs/{run_id}/jobs", repo=repo)))\n        plans: set[int] = set()\n        for repo in self.plan_repos:\n            plans |= set(self._seqs(self._list(f"runs/{run_id}/plans", repo=repo)))\n        return sorted(jobs - plans)\n\n    def get_job(self, run_id: str, seq: int) -> bytes:\n        data = self._download(f"runs/{run_id}/jobs/{seq:05d}.npz", mutable=False,\n                              repo=self.job_repos[seq % N_SHARDS])\n        if data is None:\n            raise FileNotFoundError(f"runs/{run_id}/jobs/{seq:05d}.npz")\n        return data\n\n    def put_plan(self, run_id: str, seq: int, text: str) -> None:\n        self._upload(f"runs/{run_id}/plans/{seq:05d}.json", text.encode("utf-8"),\n                     f"{run_id} plan {seq}", repo=self.plan_repos[seq % N_SHARDS])\n\n    def write_heartbeat(self, state: dict[str, Any]) -> None:\n        doc = dict(state, ts=time.time())\n        self._upload("worker/heartbeat.json", json.dumps(doc).encode("utf-8"), "heartbeat")\n\n    def stop_requested(self) -> bool:\n        return any(p.endswith("/STOP") for p in self._list("worker"))\n\n    def clear_stop(self) -> None:\n        try:\n            _retry(\n                lambda: self.api.delete_file(\n                    "worker/STOP", self.repo_id, repo_type="dataset", commit_message="clear STOP"\n                ),\n                what="delete STOP",\n            )\n        except Exception:\n            pass\n\n\nclass LocalDirQueue:\n    """Same interface as `LoopQueue`, backed by a directory. For tests and\n    for smoke-testing the CARLA side against `scripts/fake_loop_worker.py`\n    without Kaggle or a Hugging Face login. Selected via repo id `dir:<path>`."""\n\n    def __init__(self, root: str | Path) -> None:\n        self.repo_id = f"dir:{root}"\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n\n    def ensure_repo(self) -> None:\n        self.root.mkdir(parents=True, exist_ok=True)\n\n    def whoami(self) -> str:\n        return "local"\n\n    def _write(self, rel: str, data: bytes) -> None:\n        path = self.root / rel\n        path.parent.mkdir(parents=True, exist_ok=True)\n        tmp = path.with_suffix(path.suffix + ".part")\n        tmp.write_bytes(data)\n        tmp.replace(path)  # atomic on the same filesystem: readers never see partial files\n\n    def _read(self, rel: str) -> bytes | None:\n        path = self.root / rel\n        return path.read_bytes() if path.exists() else None\n\n    def _list(self, rel: str) -> list[str]:\n        d = self.root / rel\n        if not d.is_dir():\n            return []\n        return [f"{rel}/{p.name}" for p in d.iterdir() if not p.name.endswith(".part")]\n\n    _seqs = staticmethod(LoopQueue._seqs)\n\n    def start_run(self, run_id: str, info: dict[str, Any]) -> None:\n        doc = dict(info, run_id=run_id, status="active", started_at=time.time())\n        self._write(f"runs/{run_id}/run.json", json.dumps(doc, indent=1).encode("utf-8"))\n\n    def finish_run(self, run_id: str, summary: dict[str, Any] | None = None) -> None:\n        self._write(f"runs/{run_id}/DONE", json.dumps(summary or {}).encode("utf-8"))\n\n    def put_job(self, run_id: str, seq: int, payload: bytes) -> None:\n        self._write(f"runs/{run_id}/jobs/{seq:05d}.npz", payload)\n\n    def get_plan(self, run_id: str, seq: int) -> str | None:\n        data = self._read(f"runs/{run_id}/plans/{seq:05d}.json")\n        return None if data is None else data.decode("utf-8")\n\n    def heartbeat(self) -> dict[str, Any] | None:\n        data = self._read("worker/heartbeat.json")\n        return None if data is None else json.loads(data.decode("utf-8"))\n\n    def request_stop(self) -> None:\n        self._write("worker/STOP", b"stop")\n\n    def list_runs(self) -> list[str]:\n        return sorted(p.rsplit("/", 1)[-1] for p in self._list("runs"))\n\n    def run_is_done(self, run_id: str) -> bool:\n        return (self.root / f"runs/{run_id}/DONE").exists()\n\n    def pending_jobs(self, run_id: str) -> list[int]:\n        jobs = set(self._seqs(self._list(f"runs/{run_id}/jobs")))\n        plans = set(self._seqs(self._list(f"runs/{run_id}/plans")))\n        return sorted(jobs - plans)\n\n    def get_job(self, run_id: str, seq: int) -> bytes:\n        data = self._read(f"runs/{run_id}/jobs/{seq:05d}.npz")\n        if data is None:\n            raise FileNotFoundError(f"runs/{run_id}/jobs/{seq:05d}.npz")\n        return data\n\n    def put_plan(self, run_id: str, seq: int, text: str) -> None:\n        self._write(f"runs/{run_id}/plans/{seq:05d}.json", text.encode("utf-8"))\n\n    def write_heartbeat(self, state: dict[str, Any]) -> None:\n        self._write("worker/heartbeat.json", json.dumps(dict(state, ts=time.time())).encode("utf-8"))\n\n    def stop_requested(self) -> bool:\n        return (self.root / "worker/STOP").exists()\n\n    def clear_stop(self) -> None:\n        try:\n            (self.root / "worker/STOP").unlink()\n        except FileNotFoundError:\n            pass\n\n\ndef open_queue(repo_id: str, cache_dir: str | Path | None = None) -> LoopQueue | LocalDirQueue:\n    """`dir:<path>` -> LocalDirQueue, anything else -> Hugging Face repo."""\n    if repo_id.startswith("dir:"):\n        return LocalDirQueue(repo_id[4:])\n    return LoopQueue(repo_id, cache_dir=cache_dir)\n'}
for rel, src in files.items():
    p = PKG / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(src, encoding='utf-8')
try:
    import json as _json
    from huggingface_hub import HfApi, hf_hub_download
    _base = os.environ.get('LOOP_REPO', 'acarla-loop')
    if '/' not in _base: _base = HfApi().whoami()['name'] + '/' + _base
    _f = hf_hub_download(_base, 'code/acarla_pkg.json', repo_type='dataset', force_download=True)
    _files = _json.loads(Path(_f).read_text(encoding='utf-8'))
    for rel, src in _files.items():
        p = PKG / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(src, encoding='utf-8')
    print('acarla mini package refreshed from', _base, '(', len(_files), 'files )')
except Exception as _exc:
    print('no code refresh from the queue repo (using embedded package):', type(_exc).__name__, _exc)
print('acarla mini package:', sorted(str(p.relative_to(PKG)) for p in PKG.rglob('*.py')))
run([str(PYTHON312), '-c', "import sys; sys.path.insert(0, '/kaggle/working/acarla_pkg'); import acarla.loop.codec, acarla.loop.transport, huggingface_hub, PIL; print({'huggingface_hub': huggingface_hub.__version__, 'pillow': PIL.__version__})"])


In [ ]:
runner = 'from __future__ import annotations\n\nimport json\nimport os\nimport platform\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom importlib import metadata\nfrom pathlib import Path\n\nimport numpy as np\nsys.path.insert(0, \'/kaggle/working/acarla_pkg\')\n\nMODEL_ID = os.environ.get(\'M0_MODEL_ID\', \'nvidia/Alpamayo-1.5-10B\')\nUPSTREAM_COMMIT = \'24179cfa8b2eeaf775e9e21698b23af0f899522d\'\nREPO = Path(\'/kaggle/temp/alpamayo_m0/alpamayo1.5\')\nRESULT = Path(\'/kaggle/working/m0_results/m0_kaggle_attempt.json\')\nCACHE = Path(\'/kaggle/temp/huggingface\')\nsys.path.insert(0, str(REPO / \'src\'))\nos.environ[\'HF_HOME\'] = str(CACHE)\nos.environ[\'HUGGINGFACE_HUB_CACHE\'] = str(CACHE / \'hub\')\n# v19: Fragmentierung -- der OOM in v18 meldete 376 MiB reserviert-unbelegt bei 292 MiB Bedarf.\nos.environ.setdefault(\'PYTORCH_CUDA_ALLOC_CONF\', \'expandable_segments:True\')\n\ndef version(name):\n    try:\n        return metadata.version(name)\n    except metadata.PackageNotFoundError:\n        return None\n\ndef safe(value):\n    if isinstance(value, dict):\n        return {str(k): safe(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [safe(v) for v in value]\n    if isinstance(value, (str, int, float, bool)) or value is None:\n        return value\n    current = value\n    for method in (\'detach\', \'cpu\'):\n        fn = getattr(current, method, None)\n        if fn is not None:\n            current = fn()\n    fn = getattr(current, \'numpy\', None)\n    array = fn() if fn is not None else np.asarray(current)\n    return array.item() if array.ndim == 0 else array.tolist()\n\ndef write_result(result):\n    RESULT.parent.mkdir(parents=True, exist_ok=True)\n    RESULT.write_text(json.dumps(safe(result), indent=2), encoding=\'utf-8\')\n\ndef synthetic_contract_data(torch):\n    """Valid Alpamayo tensor shapes without any gated dataset content."""\n    height, width = 320, 576\n    image_frames = torch.zeros((4, 4, 3, height, width), dtype=torch.uint8)\n    x = torch.linspace(0, 255, width, dtype=torch.uint8).view(1, 1, width)\n    y = torch.linspace(0, 255, height, dtype=torch.uint8).view(1, height, 1)\n    image_frames[:, :, 0] = x\n    image_frames[:, :, 1] = y\n    for camera in range(4):\n        image_frames[camera, :, 2].fill_(camera * 60)\n    identity = torch.eye(3, dtype=torch.float32)\n    return {\n        \'image_frames\': image_frames,\n        \'camera_indices\': torch.tensor([0, 1, 2, 6], dtype=torch.int64),\n        \'ego_history_xyz\': torch.zeros((1, 1, 16, 3), dtype=torch.float32),\n        \'ego_history_rot\': identity.view(1, 1, 1, 3, 3).repeat(1, 1, 16, 1, 1),\n        \'ego_future_xyz\': torch.zeros((1, 1, 64, 3), dtype=torch.float32),\n        \'ego_future_rot\': identity.view(1, 1, 1, 3, 3).repeat(1, 1, 64, 1, 1),\n    }\n\nresult = {\n    \'status\': \'started\',\n    \'model_id\': MODEL_ID,\n    \'upstream_git_revision\': subprocess.check_output(\n        [\'git\', \'-C\', str(REPO), \'rev-parse\', \'HEAD\'], text=True\n    ).strip(),\n    \'expected_upstream_revision\': UPSTREAM_COMMIT,\n    \'attention_backend\': \'sdpa\',\n    \'seed\': 42,\n    \'repeats\': 2,\n    \'num_traj_samples\': 1,\n    \'environment\': {\n        \'python\': platform.python_version(),\n        \'torch\': version(\'torch\'),\n        \'transformers\': version(\'transformers\'),\n        \'accelerate\': version(\'accelerate\'),\n        \'hf_token_present\': bool(os.environ.get(\'HF_TOKEN\')),\n        \'hf_token_source\': os.environ.get(\'M0_SECRET_SOURCE\', \'none\'),\n    },\n}\nwrite_result(result)\n\ntry:\n    import torch\n    from alpamayo1_5 import helper\n    from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset\n    from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is unavailable\')\n    gpus = []\n    for index in range(torch.cuda.device_count()):\n        props = torch.cuda.get_device_properties(index)\n        gpus.append({\n            \'index\': index,\n            \'name\': props.name,\n            \'memory_gib\': props.total_memory / 1024**3,\n            \'compute_capability\': [props.major, props.minor],\n        })\n        torch.cuda.reset_peak_memory_stats(index)\n    result[\'environment\'][\'cuda\'] = torch.version.cuda\n    result[\'environment\'][\'gpus\'] = gpus\n    result[\'environment\'][\'driver\'] = subprocess.run(\n        [\'nvidia-smi\', \'--query-gpu=driver_version\', \'--format=csv,noheader\'],\n        capture_output=True,\n        text=True,\n        check=False,\n    ).stdout.strip().splitlines()\n\n    largest = max(gpu[\'memory_gib\'] for gpu in gpus)\n    aggregate = sum(gpu[\'memory_gib\'] for gpu in gpus)\n    native_bf16 = any(gpu[\'compute_capability\'][0] >= 8 for gpu in gpus)\n    # v20: Speicherstrategie 1 aus dem Projektauftrag. FP16 auf 2x T4 ist widerlegt:\n    # v18 (12/12 GiB) -> OOM auf GPU 1 im SDPA des Reasoning-Rollouts; v19 (12/9 GiB) -> 25 Module\n    # auf CPU ausgelagert und Cross-Device-cat im Upstream-Rollout. FP16-Gewichte ~24 GB auf 29 GB\n    # lassen kein Polster fuer Aktivierungen. NF4 nur auf den LM-Layern des Backbones; Vision-Encoder,\n    # lm_head und der gesamte Diffusions-Expert (der die Zahlen ausgibt) bleiben FP16.\n    requested = os.environ.get(\'M0_STRATEGY\', \'nf4\')\n    try:\n        import bitsandbytes  # noqa: F401\n        bnb_available = True\n    except Exception as exc:\n        bnb_available = False\n        result[\'bitsandbytes_import_error\'] = f\'{type(exc).__name__}: {exc}\'\n    if largest >= 24 and native_bf16:\n        strategy = \'upstream_bf16_single_gpu_sdpa\'\n        dtype = torch.bfloat16\n        load_kwargs = {}\n    elif requested == \'nf4\' and bnb_available and largest >= 14:\n        from transformers import BitsAndBytesConfig\n        # v21: NF4 auf einer GPU (v20) lief in den OOM bei 13.99 GiB -- die FP16-Reste (Embedding,\n        # lm_head, Vision, Expert 4.6 GB) summieren sich auf ~12 GB Gewichte. Deshalb ueber alle GPUs\n        # verteilen, aber OHNE cpu in max_memory: kein stilles Offload (v19 zeigte, dass der\n        # Upstream-Rollout damit nicht laeuft) -- lieber laut scheitern.\n        multi_gpu = len(gpus) >= 2 and os.environ.get(\'M0_NF4_SINGLE_GPU\') != \'1\'\n        # v22: device_map=\'auto\' (v21) schneidet layerweise; der Expert liest den KV-Cache ALLER\n        # VLM-Layer und konkatenierte dann ueber die Geraetegrenze (cache_utils.update, torch.cat).\n        # Deshalb explizit entlang der Datenabhaengigkeit teilen: alle LM-Layer + Expert auf GPU 1\n        # (Cache und Leser auf derselben Karte), Vision + Embedding + lm_head auf GPU 0 (die\n        # 151k-Vokabular-Logits entstehen dort, wo sonst Platz waere).\n        strategy = (\'experimental_nf4_backbone_fp16_expert_kvlocal_split_v23\' if multi_gpu\n                    else \'experimental_nf4_backbone_fp16_expert_single_gpu_v20\')\n        kv_local_map = {\n            \'vlm.model.visual\': 0,\n            \'vlm.model.language_model.embed_tokens\': 0,\n            # v23: lm_head zu den Layern -- der Rollout haelt Logits, Sequenzen und den\n            # Diffusionszustand auf input_ids.device (alpamayo1_5.py:286); alles ab Layer 0\n            # muss deshalb auf EINER Karte liegen. v22 scheiterte in _euler an x(GPU0)+v(GPU1).\n            \'vlm.lm_head\': 1,\n            \'vlm.model.language_model.layers\': 1,\n            \'vlm.model.language_model.norm\': 1,\n            \'vlm.model.language_model.rotary_emb\': 1,\n            \'expert\': 1, \'action_space\': 1, \'diffusion\': 1,\n            \'action_in_proj\': 1, \'action_out_proj\': 1,\n        }\n        dtype = torch.float16\n        skip_modules = [\n            \'visual\',            # vlm.model.visual -- Vision-Encoder\n            \'lm_head\',           # vlm.lm_head -- Reasoning-Token-Logits\n            \'expert\',            # Diffusions-/Flow-Matching-Expert: gibt die Trajektorie aus\n            \'diffusion\', \'action_space\', \'action_in_proj\', \'action_out_proj\',\n        ]\n        load_kwargs = {\n            \'device_map\': kv_local_map if multi_gpu else {\'\': 0},\n            \'quantization_config\': BitsAndBytesConfig(\n                load_in_4bit=True,\n                bnb_4bit_quant_type=\'nf4\',\n                bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=torch.float16,\n                llm_int8_skip_modules=skip_modules,\n            ),\n        }\n        result[\'quantization\'] = {\n            \'scheme\': \'nf4_double_quant\', \'compute_dtype\': \'float16\',\n            \'skip_modules\': skip_modules,\n        }\n    elif len(gpus) >= 2 and aggregate >= 24:\n        strategy = \'experimental_fp16_device_map_auto_sdpa_asym_v19\'\n        dtype = torch.float16\n        # v19: asymmetrische Kappen. In v18 (12/12 GiB) landeten LM-Layer 20-35, lm_head und der\n        # komplette Diffusions-Expert auf der LETZTEN GPU, die dann bei der Inferenz (Logits ueber das\n        # volle Vokabular, SDPA im Reasoning-Rollout) mit 14.48/14.56 GiB in den OOM lief. Die letzte\n        # GPU bekommt deshalb 3 GiB weniger Gewichte, damit dort Platz fuer Aktivierungen bleibt.\n        last_index = max(gpu[\'index\'] for gpu in gpus)\n        max_memory = {\n            gpu[\'index\']: f"{max(1, int(gpu[\'memory_gib\'] - (5 if gpu[\'index\'] == last_index else 2)))}GiB"\n            for gpu in gpus\n        }\n        max_memory[\'cpu\'] = \'24GiB\'\n        load_kwargs = {\n            \'device_map\': \'auto\',\n            \'max_memory\': max_memory,\n            \'offload_folder\': \'/kaggle/temp/alpamayo_offload\',\n            \'offload_state_dict\': True,\n        }\n    else:\n        result[\'status\'] = \'blocked_by_hardware\'\n        result[\'strategy\'] = \'none\'\n        result[\'block_reason\'] = (\n            \'Need one 24 GiB sm80+ GPU or at least two GPUs with 24 GiB aggregate; \'\n            f"found {len(gpus)} GPU(s), {aggregate:.1f} GiB."\n        )\n        raise SystemExit(2)\n    result[\'strategy\'] = strategy\n    result[\'model_dtype\'] = str(dtype)\n    result[\'strategy_classification\'] = (\n        \'reference\' if strategy.startswith(\'upstream_\') else \'experimental_unverified\'\n    )\n    result[\'max_memory\'] = load_kwargs.get(\'max_memory\')\n    write_result(result)\n\n\n    # ================= LOOP-WORKER: Jobs aus dem HF-Dataset-Repo bedienen ========================\n    import hashlib\n    from acarla.loop.codec import decode_job, encode_plan\n    from acarla.loop.transport import LoopQueue\n    LOOP_REPO = os.environ.get(\'LOOP_REPO\', \'acarla-loop\')\n    IDLE_EXIT_S = float(os.environ.get(\'LOOP_IDLE_EXIT_MIN\', \'25\')) * 60.0\n    POLL_S = float(os.environ.get(\'LOOP_POLL_S\', \'3\'))\n    HEARTBEAT_S = 300.0   # nur im Leerlauf; jeder Heartbeat ist ein Commit (128/h/Repo)\n    session_id = time.strftime(\'%Y%m%d-%H%M%S\') + \'-\' + os.environ.get(\'KAGGLE_KERNEL_RUN_TYPE\', \'run\')\n    q = LoopQueue(LOOP_REPO, cache_dir=\'/kaggle/temp/loop_cache\')\n    result[\'loop\'] = {\'repo\': LOOP_REPO, \'session\': session_id, \'idle_exit_s\': IDLE_EXIT_S}\n    hb_state = {\'session\': session_id, \'state\': \'loading\', \'model_loaded\': False, \'n_done\': 0,\n                \'current\': None, \'strategy\': strategy}\n    last_hb = [0.0]\n    def heartbeat(**changes):\n        hb_state.update(changes)\n        try:\n            q.write_heartbeat(hb_state); last_hb[0] = time.time()\n        except Exception as exc:\n            print(\'heartbeat failed:\', exc, flush=True)\n    q.ensure_repo()\n    heartbeat(state=\'loading\')\n    write_result(result)\n\n    load_start = time.perf_counter()\n    model = Alpamayo1_5.from_pretrained(\n        MODEL_ID,\n        dtype=dtype,\n        attn_implementation=\'sdpa\',\n        **load_kwargs,\n    )\n    if not load_kwargs:\n        model = model.to(\'cuda\')\n    model.eval()\n    result[\'model_load_seconds\'] = time.perf_counter() - load_start\n    result[\'hf_device_map\'] = safe(getattr(model, \'hf_device_map\', None))\n    result[\'parameter_devices\'] = sorted({str(p.device) for p in model.parameters()})\n    quantized = sorted({n.rsplit(\'.\', 1)[0] for n, m in model.named_modules()\n                        if type(m).__name__ in (\'Linear4bit\', \'Linear8bitLt\')})\n    result[\'quantized_module_count\'] = len(quantized)\n    result[\'quantized_modules_sample\'] = quantized[:6] + ([\'...\'] if len(quantized) > 6 else [])\n    result[\'expert_dtypes\'] = sorted({str(p.dtype) for n, p in model.named_parameters() if n.startswith(\'expert\')})\n    result[\'lm_layer_dtypes\'] = sorted({str(p.dtype) for n, p in model.named_parameters()\n                                        if \'language_model.layers.\' in n})\n\n    processor = helper.get_processor(model.tokenizer)\n    input_device = next(model.expert.parameters()).device\n    result[\'input_device\'] = str(input_device)\n    sampling = {\'seed\': 42, \'top_p\': 0.98, \'temperature\': 0.6, \'num_traj_samples\': 1, \'max_generation_length\': 256}\n    cfg_str = json.dumps({\'strategy\': strategy, \'revision\': result[\'upstream_git_revision\'], \'dtype\': str(dtype),\n                          \'sampling\': sampling, \'quantization\': result.get(\'quantization\')}, sort_keys=True)\n    model_config_hash = hashlib.sha256(cfg_str.encode()).hexdigest()[:16]\n    result[\'model_config_hash\'] = model_config_hash\n    result[\'sampling\'] = sampling\n    write_result(result)\n    print(json.dumps({\'model_loaded_s\': result.get(\'model_load_seconds\'), \'config_hash\': model_config_hash}), flush=True)\n\n    def run_job(run_id, seq):\n        job = decode_job(q.get_job(run_id, seq))\n        assert tuple(job[\'camera_indices\'].tolist()) == (0, 1, 2, 6), job[\'camera_indices\']\n        assert job[\'image_frames\'].shape[:3] == (4, 4, 3), job[\'image_frames\'].shape\n        assert job[\'ego_history_xyz\'].shape == (1, 1, 16, 3), job[\'ego_history_xyz\'].shape\n        frames = torch.from_numpy(job[\'image_frames\'])\n        cam_idx = torch.from_numpy(job[\'camera_indices\'])\n        messages = helper.create_message(frames=frames.flatten(0, 1), camera_indices=cam_idx)\n        inputs = processor.apply_chat_template(\n            messages, tokenize=True, add_generation_prompt=False,\n            continue_final_message=True, return_dict=True, return_tensors=\'pt\',\n        )\n        model_inputs = helper.to_device({\n            \'tokenized_data\': inputs,\n            \'ego_history_xyz\': torch.from_numpy(job[\'ego_history_xyz\']),\n            \'ego_history_rot\': torch.from_numpy(job[\'ego_history_rot\']),\n        }, input_device)\n        torch.manual_seed(sampling[\'seed\']); torch.cuda.manual_seed_all(sampling[\'seed\'])\n        torch.cuda.synchronize(); start = time.perf_counter()\n        with torch.inference_mode(), torch.autocast(\'cuda\', dtype=dtype):\n            pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(\n                data=model_inputs, top_p=sampling[\'top_p\'], temperature=sampling[\'temperature\'],\n                num_traj_samples=sampling[\'num_traj_samples\'],\n                max_generation_length=sampling[\'max_generation_length\'], return_extra=True,\n            )\n        torch.cuda.synchronize(); dt_s = time.perf_counter() - start\n        xyz = pred_xyz.detach().float().cpu().numpy()[0, 0, 0]\n        rot = pred_rot.detach().float().cpu().numpy()[0, 0, 0]\n        cot = None\n        try:\n            cot = safe(extra)[\'cot\'][0][0][0]\n        except Exception:\n            pass\n        text = encode_plan(\n            run_id=run_id, seq=seq, frame_id=job[\'meta\'][\'frame_id\'], sim_time=job[\'meta\'][\'sim_time\'],\n            waypoints_xyz=xyz, waypoints_rot=rot, reasoning=cot, inference_ms=dt_s * 1000.0,\n            model_config_hash=model_config_hash, worker_session=session_id,\n            extra={\'codec\': job[\'meta\'].get(\'codec\'), \'strategy\': strategy},\n        )\n        q.put_plan(run_id, seq, text)\n        del model_inputs, inputs, pred_xyz, pred_rot, extra\n        torch.cuda.empty_cache()\n        return dt_s, cot, xyz[-1]\n\n    heartbeat(state=\'idle\', model_loaded=True)\n    done_runs = set()\n    last_job_t = time.time()\n    n_done = 0\n    n_failed = 0\n    exit_reason = None\n    while True:\n        if q.stop_requested():\n            q.clear_stop(); exit_reason = \'stop_requested\'; break\n        work = None\n        for run_id in reversed(q.list_runs()):          # neuester Lauf zuerst\n            if run_id in done_runs:\n                continue\n            if q.run_is_done(run_id):\n                done_runs.add(run_id); continue\n            pending = q.pending_jobs(run_id)\n            if pending:\n                work = (run_id, pending[0]); break\n        if work is None:\n            idle_s = time.time() - last_job_t\n            if idle_s > IDLE_EXIT_S:\n                exit_reason = \'idle_timeout\'; break\n            if time.time() - last_hb[0] > HEARTBEAT_S:\n                heartbeat(state=\'idle\', current=None, idle_s=round(idle_s))\n            time.sleep(POLL_S); continue\n        run_id, seq = work\n        hb_state.update(state=\'busy\', current={\'run\': run_id, \'seq\': seq})\n        try:\n            dt_s, cot, end = run_job(run_id, seq)\n            n_done += 1; last_job_t = time.time()\n            print(json.dumps({\'run\': run_id, \'seq\': seq, \'inference_s\': round(dt_s, 1),\n                              \'end_xyz\': [round(float(v), 2) for v in end], \'cot\': cot}), flush=True)\n        except Exception as exc:\n            n_failed += 1\n            print(json.dumps({\'run\': run_id, \'seq\': seq, \'error\': f\'{type(exc).__name__}: {exc}\'}), flush=True)\n            traceback.print_exc()\n            # Fehlerantwort schreiben, damit die lokale Seite nicht ewig wartet (finite=false)\n            try:\n                bad = np.full((64, 3), np.nan); eye = np.tile(np.eye(3), (64, 1, 1))\n                q.put_plan(run_id, seq, encode_plan(\n                    run_id=run_id, seq=seq, frame_id=-1, sim_time=-1.0, waypoints_xyz=bad, waypoints_rot=eye,\n                    reasoning=f\'WORKER ERROR {type(exc).__name__}: {exc}\', inference_ms=0.0,\n                    model_config_hash=model_config_hash, worker_session=session_id))\n            except Exception as exc2:\n                print(\'could not write error plan:\', exc2, flush=True)\n            if n_failed >= 5 and n_done == 0:\n                exit_reason = \'repeated_failures\'; break\n        hb_state.update(n_done=n_done, n_failed=n_failed, state=\'idle\', current=None)  # Plan-Upload belegt Leben\n        result[\'loop\'].update({\'n_done\': n_done, \'n_failed\': n_failed}); write_result(result)\n\n    heartbeat(state=\'exited\', exit_reason=exit_reason, current=None, n_done=n_done)\n    result[\'loop\'].update({\'exit_reason\': exit_reason, \'n_done\': n_done, \'n_failed\': n_failed})\n    result[\'cuda_memory\'] = [{\n        \'index\': index,\n        \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(index),\n        \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(index),\n    } for index in range(torch.cuda.device_count())]\n    result[\'status\'] = \'ok\'\nexcept SystemExit:\n    raise\nexcept Exception as exc:\n    result[\'status\'] = \'failed\'\n    result[\'exception\'] = {\'type\': type(exc).__name__, \'message\': str(exc), \'traceback\': traceback.format_exc()}\n    try:\n        heartbeat(state=\'failed\', error=f\'{type(exc).__name__}: {exc}\')\n    except Exception:\n        pass\nfinally:\n    write_result(result)\n    print(json.dumps({\'status\': result.get(\'status\'), \'strategy\': result.get(\'strategy\'),\n                      \'loop\': result.get(\'loop\'), \'exception\': (result.get(\'exception\') or {}).get(\'message\')}, indent=2))\nif result.get(\'status\') == \'failed\':\n    raise SystemExit(1)\n'
runner_path = Path('/kaggle/working/alpamayo_m0_runner.py')
runner_path.write_text(runner, encoding='utf-8')
print(f'Wrote {runner_path} ({len(runner.splitlines())} lines)')

In [ ]:
completed = subprocess.run([str(PYTHON312), str(runner_path)], env=os.environ.copy(), check=False)
print('Runner exit code:', completed.returncode)
if RESULT.exists():
    m0 = json.loads(RESULT.read_text(encoding='utf-8'))
    display({k: m0.get(k) for k in ('status', 'strategy', 'model_load_seconds', 'model_config_hash', 'loop', 'exception')})
